In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/mestrado/PoiMtlNet')
DATA_ROOT = DRIVE_ROOT / 'data'
OUTPUT_DIR = DRIVE_ROOT / 'output'
RESULTS_ROOT = DRIVE_ROOT / 'results'

for p in [DATA_ROOT, OUTPUT_DIR, RESULTS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['DATA_ROOT'] = str(DATA_ROOT)
os.environ['OUTPUT_DIR'] = str(OUTPUT_DIR)
os.environ['RESULTS_ROOT'] = str(RESULTS_ROOT)

print(f'DATA_ROOT: {DATA_ROOT}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'RESULTS_ROOT: {RESULTS_ROOT}')


In [ ]:
REPO_DIR = Path('/content/PoiMtlNet')

if not REPO_DIR.exists():
    get_ipython().system(f'git clone https://github.com/VitorHugoOli/PoiMtlNet.git {REPO_DIR}')
else:
    get_ipython().system(f'git -C {REPO_DIR} pull')

%cd {REPO_DIR}
!git pull
!git log --oneline -3


In [ ]:
!pip install -q -r requirements_colab.txt
!pip install -q cvxpy
!pip install -q torch-geometric
!pip install -q pytorch_warmup
!pip install -q torchmetrics
!pip install -q fvcore

import torch

!pip uninstall torch-scatter torch-sparse torch-geometric torch-cluster -y
!pip install torch-scatter -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-cluster -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install git+https://github.com/pyg-team/pytorch_geometric.git
!pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu121


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/PoiMtlNet')

STATES = ['alabama', 'florida', 'california', 'texas']
EMBEDDINGS = ['sphere2vec', 'time2vec', 'hgi']

for state in STATES:
    print(f'\n=== STATE: {state.upper()} ===')
    procs = []

    for engine in EMBEDDINGS:
        cmd = [
            sys.executable,
            str(REPO_DIR / 'scripts' / 'run_single_embedding.py'),
            '--state',
            state,
            '--engine',
            engine,
        ]
        p = subprocess.Popen(cmd, cwd=REPO_DIR)
        procs.append((engine, p))
        print(f'Started {engine} for {state} (pid={p.pid})')

    failures = []
    for engine, p in procs:
        rc = p.wait()
        if rc == 0:
            print(f'OK: {engine} | {state}')
        else:
            print(f'FAIL: {engine} | {state} (exit={rc})')
            failures.append((engine, rc))

    if failures:
        raise RuntimeError(f'Embedding failures for state {state}: {failures}')

print('\nAll requested embeddings finished for all states.')
